# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

metadata = dataset.metadata.to_json()
print("Dataset Name:", metadata.get('name', 'N/A'))
print("Description:", metadata.get('description', 'N/A'))
print("Published:", metadata.get('datePublished', 'N/A'))

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their fields using their @id
print("Available Record Sets and their fields (@id):\n")
record_sets = list(dataset.metadata.record_sets)
for rs in record_sets:
    print(f"RecordSet: {rs['@id']}  Name: {rs.get('name', 'N/A')}")
    print("  Fields:")
    for field in rs.get('fields', []):
        print(f"    Field @id: {field['@id']} | Name: {field.get('name', 'N/A')}")
    print('-' * 60)
# Optionally, print columns if present
    if 'columns' in rs:
        print("  Columns:")
        for col in rs.get('columns', []):
            print(f"    Column @id: {col['@id']} | Name: {col.get('name', 'N/A')}")
        print('-' * 60)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Example: Let us extract records from the primary survey record set.
# Identify the main record set (likely to have GAD-7, PHQ-9 and demographics)

# List all record set @id values for reference
record_sets = list(dataset.metadata.record_sets)
record_set_ids = [rs['@id'] for rs in record_sets]
print('Record Sets in this dataset:', record_set_ids)

# Usually, the main survey set will have 'survey', 'responses', or similar in the name (here we choose the first as a demonstration)
main_record_set_id = record_set_ids[0] if len(record_set_ids) else None
print(f"Chosen Record Set for extraction: {main_record_set_id}")

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Extracting records for {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records\n")

if main_record_set_id:
    print('Columns for main record set:', dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Let us choose a numeric field (e.g., GAD-7 total score, PHQ-9 total score)
# To ensure the code is robust, let's try to pick 'gad7_total' or 'phq9_total' if present

numeric_candidates = ['gad7_total', 'phq9_total', 'psq_total', 'score_gad7', 'score_phq9']

main_df = dataframes[main_record_set_id]
numeric_field = None
for candidate in numeric_candidates:
    if candidate in main_df.columns:
        numeric_field = candidate
        break

if numeric_field is None:
    # Try to use the first numeric column
    for col in main_df.columns:
        # Try and detect numerical columns
        if pd.api.types.is_numeric_dtype(main_df[col]):
            numeric_field = col
            break

print(f"Using numeric field: {numeric_field}")

# Choose a threshold for filtering
threshold = 10
filtered_df = main_df[main_df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the field
filtered_df[numeric_field + '_normalized'] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()

print(f"Normalized '{numeric_field}' for filtered records:")
print(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

# Group by a categorical field, e.g., 'gender', 'marital_status', 'village', or similar
group_candidates = ['gender', 'marital_status', 'village', 'level_of_education']
group_field = None
for candidate in group_candidates:
    if candidate in main_df.columns:
        group_field = candidate
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped data by {group_field}, mean of {numeric_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Histogram of numeric field
plt.figure(figsize=(8,5))
main_df[numeric_field].hist(bins=20, color='skyblue', edgecolor='black')
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.grid(axis='y')
plt.show()

# Boxplot by group (if available)
if group_field:
    plt.figure(figsize=(10,6))
    main_df.boxplot(column=numeric_field, by=group_field, grid=False)
    plt.title(f'{numeric_field} by {group_field}')
    plt.suptitle("")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* In this notebook, we explored the FAIR² Mental Health Survey Dataset using `mlcroissant` and Pandas.
* We identified record sets, loaded records into DataFrames, and explored numerical fields relevant to mental health assessment.
* Basic filtering, normalization, grouping, and visualizations illustrated key statistics and potential directions for deeper analysis.
* For more extensive studies, consider examining missing data handling, longitudinal analysis, and cross-field relationships (e.g., demographic predictors of GAD-7/PHQ-9 scores).